# Documentação técnica — Trilha SEFAZ-SP

Este notebook reúne a visão funcional e técnica do projeto, descreve como os módulos se conectam e contém verificações executáveis do código-fonte. Ele complementa o `README.md`; não substitui os materiais de estudo nem os arquivos de configuração externos.

## Objetivos

- Entender a arquitetura do site e o fluxo de inicialização.
- Documentar as responsabilidades de HTML, CSS, JavaScript, Python e Supabase.
- Explicar o pipeline dos arquivos Markdown até os materiais JSON privados.
- Identificar comandos de desenvolvimento, validação e publicação.
- Rodar inspeções somente de metadados e código, sem abrir conteúdos didáticos privados.

> **Privacidade:** os diretórios de disciplinas e `data/` são ignorados pelo Git. Não publique materiais, credenciais, senhas ou tokens privados. A chave Supabase publicável é destinada ao navegador; nunca inclua uma chave `service_role` ou equivalente.

## Como usar

Execute as células Python em ordem, com um kernel Python local. As células de inspeção procuram o diretório do projeto a partir da pasta de trabalho atual e exibem apenas nomes de arquivos, assinaturas de funções, contagens e metadados — nunca o texto dos materiais.

In [1]:
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "extractor.py").is_file() and (path / "app.js").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Não encontrei a raiz do projeto. Abra este notebook na pasta que contém app.js e extractor.py."
    )

CORE_FILES = (
    "index.html",
    "styles.css",
    "app.js",
    "extractor.py",
    "supabase-config.js",
    "supabase-schema.sql",
    "README.md",
)
missing_files = [name for name in CORE_FILES if not (PROJECT_ROOT / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Arquivos principais ausentes: {', '.join(missing_files)}")

print(f"Raiz do projeto: {PROJECT_ROOT}")
print("Arquivos principais verificados:")
for name in CORE_FILES:
    print(f"  - {name}")

Raiz do projeto: c:\Users\Magno Santos\OneDrive\Desktop\ESTUDO\_Estrategia_SEFAZ-SP_2025\zip\extraidos
Arquivos principais verificados:
  - index.html
  - styles.css
  - app.js
  - extractor.py
  - supabase-config.js
  - supabase-schema.sql
  - README.md


## 1. Visão geral e arquitetura

A Trilha é uma aplicação estática para estudos, composta pelo navegador (HTML, CSS e JavaScript), um pré-processador Python para materiais Markdown e serviços Supabase para autenticação, armazenamento privado de materiais e sincronização do progresso.

```text
Fontes Markdown locais
        │
        ▼
extractor.py ──► manifest.json + subjects/*.json + question-index.json
                                      │ (upload manual para bucket privado)
                                      ▼
Navegador: index.html + styles.css + app.js ──► Supabase Auth / Storage / user_progress
                                      │
                                      └──► localStorage (cache e progresso local)
```

### Áreas funcionais

- **Visão geral:** métricas de desempenho e atividade de estudo.
- **Conteúdos:** leitor Markdown, busca/navegação por material e marcação de estudo.
- **Questões:** treino e simulados com alternativas, gabaritos, comentários e histórico.
- **Mapas mentais e flashcards:** recursos de revisão derivados dos materiais.
- **Organização dos estudos:** plano, calendário, metas, Pomodoro, revisões e diário.
- **Conta e histórico:** acesso autorizado, recuperação de senha e sincronização.

### Mapa dos arquivos

| Arquivo | Responsabilidade |
|---|---|
| `index.html` | Estrutura da página, navegação, tela de autenticação e ponto de montagem das telas. |
| `styles.css` | Layout, componentes visuais, leitor e regras responsivas. |
| `app.js` | Estado da interface, autenticação, leitura de dados, renderização e interações. |
| `extractor.py` | Limpeza dos arquivos Markdown, extração de questões, flashcards e JSONs de publicação. |
| `tests/test_extractor.py` | Testes unitários da extração e normalização dos materiais. |
| `supabase-config.js` | URL pública do projeto Supabase, chave publicável e nome do bucket. |
| `supabase-schema.sql` | Tabela de progresso, bucket privado e políticas de acesso (RLS). |
| `README.md` | Instruções de instalação, configuração e operação. |
| `.gitignore` | Exclui materiais, dados gerados e arquivos locais/sensíveis do Git. |

A célula seguinte confere arquivos e localiza as funções de entrada mais relevantes sem imprimir o código-fonte.

In [2]:
import ast
import re

source_paths = {
    "HTML": PROJECT_ROOT / "index.html",
    "CSS": PROJECT_ROOT / "styles.css",
    "JavaScript": PROJECT_ROOT / "app.js",
    "Python": PROJECT_ROOT / "extractor.py",
    "SQL": PROJECT_ROOT / "supabase-schema.sql",
}
for label, path in source_paths.items():
    print(f"{label:12} {len(path.read_text(encoding='utf-8').splitlines()):5} linhas")

app_source = source_paths["JavaScript"].read_text(encoding="utf-8")
key_functions = re.findall(
    r"(?m)^\s*(?:async\s+)?function\s+(initializeAuth|loadPrivateJson|loadSubject|setTab|renderDashboard|renderContents|renderQuestions|renderOrganization|renderMindmaps|renderFlashcards)\s*\(",
    app_source,
)
print("\nFunções principais do fluxo da aplicação:")
for name in dict.fromkeys(key_functions):
    print(f"  - {name}")

python_tree = ast.parse((PROJECT_ROOT / "extractor.py").read_text(encoding="utf-8"))
extractor_functions = [node.name for node in python_tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
print("\nEtapas públicas do extrator:")
for name in extractor_functions:
    print(f"  - {name}")

test_tree = ast.parse((PROJECT_ROOT / "tests" / "test_extractor.py").read_text(encoding="utf-8"))
test_count = sum(
    isinstance(node, ast.FunctionDef) and node.name.startswith("test_")
    for node in ast.walk(test_tree)
)
print(f"\nCasos de teste declarados: {test_count}")

HTML           100 linhas
CSS            993 linhas
JavaScript    2422 linhas
Python         613 linhas
SQL             58 linhas

Funções principais do fluxo da aplicação:
  - loadPrivateJson
  - initializeAuth
  - loadSubject
  - setTab
  - renderDashboard
  - renderOrganization
  - renderContents
  - renderQuestions
  - renderMindmaps
  - renderFlashcards

Etapas públicas do extrator:
  - stable_id
  - clean_title
  - normalize_answer
  - is_question_candidate
  - parse_options
  - normalize_question_layout
  - remove_page_counters
  - clean_study_metadata
  - clean_content_artifacts
  - clean_pdf_artifacts
  - clean_promotional_content
  - question_completeness
  - question_from_block
  - extract_questions
  - deduplicate_subject_questions
  - generate_flashcards
  - extract
  - main

Casos de teste declarados: 20


## 2. Pipeline dos materiais e contrato dos dados

`extractor.py` usa apenas a biblioteca padrão do Python. A função `extract()` percorre as subpastas imediatas da origem como disciplinas e procura `.md` recursivamente em cada uma. Para cada arquivo, o fluxo é:

1. Ler UTF-8 (com tolerância a BOM e caracteres inválidos).
2. Separar teoria e questões de múltipla escolha; identificar assunto, banca, alternativas, gabarito e comentário quando disponíveis.
3. Limpar contadores, cabeçalhos repetidos, marcação residual e publicidade de conversão PDF/OCR.
4. Remover duplicatas de questões por identificador externo dentro da disciplina, preservando a versão mais completa.
5. Gerar flashcards candidatos a partir de títulos e parágrafos; revisar a qualidade antes de usá-los como fonte definitiva.
6. Gravar JSON por disciplina, manifesto geral e índice leve de questões.

### Saídas e campos principais

- `manifest.json`: data de geração, disciplinas, relação de materiais e totais.
- `subjects/<id>.json`: objeto `subject` e lista `materials`.
- Cada material contém `id`, `title`, `source`, `content`, `questions` e `flashcards`.
- Cada questão contém `id`, `number`, `statement`, `options`, `answer`, `solution`, `source`, `topic` e `bank`.
- `question-index.json`: mapeia `id` da questão a `subjectId`, `topic` e `bank`, para consultas leves sem baixar todos os materiais.

Os arquivos em `data/` e os materiais por disciplina **não devem ser enviados ao repositório público**. Depois de gerar os JSONs, carregue-os no bucket privado `study-materials`, conservando os caminhos indicados no manifesto.

### Executar a extração

```powershell
# Usa as pastas de materiais do projeto e grava em .\data
python extractor.py

# Formato para outra origem e diretório de saída
python extractor.py "C:\caminho\para\materiais" --output ".\data"
```

A célula a seguir exercita o parser em memória, sem escrever arquivos e sem exibir texto de materiais reais.

In [3]:
import sys

project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
from extractor import extract_questions

example_markdown = """## Noções básicas\n\nQuestão 12345\nEnunciado demonstrativo.\nA) Alternativa um\nB) Alternativa dois\nGabarito: B\n"""
theory, parsed_questions = extract_questions(example_markdown, "Exemplo/aula.md")
assert len(parsed_questions) == 1, "O exemplo deveria produzir uma questão."
assert [option["key"] for option in parsed_questions[0]["options"]] == ["A", "B"]
assert parsed_questions[0]["answer"] == "B"
assert "Enunciado demonstrativo" not in theory
print("Parser em memória validado: 1 questão, 2 alternativas e gabarito identificado.")
print("Nenhum arquivo de saída foi criado nesta verificação.")

Parser em memória validado: 1 questão, 2 alternativas e gabarito identificado.
Nenhum arquivo de saída foi criado nesta verificação.


## 3. Aplicação web: navegação, renderização e estado

### Inicialização e carregamento

1. `index.html` cria o shell da aplicação e mantém a tela de autenticação visível inicialmente; os scripts são carregados com `defer`.
2. `app.js` chama `start()`, inicializa o cliente Supabase e acompanha a sessão autenticada.
3. Após validar o usuário e carregar o progresso da conta, `loadStudyMaterials()` busca `manifest.json` do bucket privado.
4. A navegação das disciplinas é montada a partir do manifesto. `loadSubject()` baixa o JSON daquela disciplina sob demanda e o armazena em cache de memória.
5. `setTab()` seleciona uma tela; funções `render*` produzem o conteúdo e conectam eventos da interface.

O manifesto funciona como índice pequeno; os arquivos grandes das disciplinas são baixados apenas quando necessários. Falhas de autenticação, rede ou leitura são exibidas na tela e registradas no console, em vez de liberar um resultado falso.

### Responsabilidades do JavaScript

- **Estado e armazenamento:** estado transitório em memória; histórico e preferências locais em `localStorage`, com chaves versionadas.
- **Autenticação e sincronização:** permite apenas a conta configurada, carrega progresso de `user_progress` e serializa salvamentos em lote com atraso curto. Uma importação explícita mescla o histórico local preservado com a conta.
- **Leitor de conteúdos:** normaliza Markdown e artefatos de PDF/OCR, renderiza títulos, listas, tabelas, sumários e chamadas visuais. Links são limitados a protocolos seguros e o conteúdo textual passa por escape HTML.
- **Questões e simulados:** navega por questões, registra resposta, correção, tempo, alternativas eliminadas e resultados de simulado.
- **Outras áreas:** dashboard, mapas mentais, flashcards e ferramentas de organização do estudo.

### Estrutura da interface e estilos

`index.html` declara a navegação principal e os elementos persistentes; `app.js` injeta a tela ativa dentro de `#view`. `styles.css` organiza os componentes por área (autenticação, navegação, cartões, conteúdo, questões, calendário e revisão), com breakpoints para adaptar sidebar, grades, tabelas e leitor a telas menores.

A célula abaixo verifica que as seis áreas da navegação têm nomes únicos e que existem regras CSS responsivas.

In [4]:
html_source = (PROJECT_ROOT / "index.html").read_text(encoding="utf-8")
css_source = (PROJECT_ROOT / "styles.css").read_text(encoding="utf-8")
nav_tabs = re.findall(r'<button class="nav-link[^\"]*" data-tab="([^"]+)"', html_source)
expected_tabs = {"dashboard", "contents", "questions", "mindmaps", "flashcards", "organization"}
assert set(nav_tabs) == expected_tabs, f"Navegação inesperada: {nav_tabs}"
assert len(nav_tabs) == len(set(nav_tabs)), "Há áreas duplicadas na navegação."
breakpoints = re.findall(r"@media\s*\(max-width:\s*([^)]+)\)", css_source)
assert breakpoints, "Nenhuma regra responsiva @media foi encontrada."
print(f"Áreas de navegação verificadas: {len(nav_tabs)} — {', '.join(nav_tabs)}")
print(f"Breakpoints responsivos CSS identificados: {', '.join(breakpoints)}")

Áreas de navegação verificadas: 6 — dashboard, contents, questions, mindmaps, flashcards, organization
Breakpoints responsivos CSS identificados: 600px, 1080px, 820px, 560px


## 4. Supabase, autenticação e privacidade

A aplicação é estática: não existe servidor próprio nem segredo de backend no navegador. A proteção dos dados depende de uma sessão Supabase válida e, principalmente, das políticas de segurança no banco e no Storage.

### Fluxo de acesso

- `supabase-config.js` contém somente a URL do projeto, a chave **publishable** e o nome do bucket. Esses valores conectam o cliente, mas não substituem autorização.
- `app.js` aceita apenas o usuário configurado, carrega `manifest.json` e JSONs de disciplina do bucket privado e sincroniza o snapshot de progresso em `user_progress`.
- `supabase-schema.sql` habilita RLS em `user_progress`, restringe operações ao usuário autorizado e cria o bucket privado com limite de arquivo.
- O progresso local continua em `localStorage`; a importação para uma conta é explícita, e falhas de sincronização são informadas.
- Os identificadores de conta usados nas verificações do cliente e nas políticas precisam ser coerentes. Alterar o usuário autorizado exige atualizar os dois lados e revisar as políticas implantadas.

### Configuração e publicação

1. Execute `supabase-schema.sql` no SQL Editor do projeto Supabase.
2. Crie a conta autorizada no painel e desative novos cadastros públicos.
3. Configure URLs de redirecionamento para o domínio publicado e para o endereço local de desenvolvimento.
4. Confira `url`, `publishableKey` e `materialsBucket` em `supabase-config.js`. Não grave senhas, `service_role`, `sb_secret_` ou tokens privados em código ou notebook.
5. Gere os JSONs com `extractor.py` e carregue `manifest.json`, `question-index.json` e `subjects/*.json` em `study-materials`, mantendo o bucket privado.
6. Publique os arquivos estáticos pelo GitHub Pages ou hospedagem equivalente. Nunca publique `data/` nem as pastas de materiais.

A validação na interface melhora a experiência; a autorização efetiva dos dados deve continuar nas políticas RLS do Supabase. O cliente do navegador não deve ser tratado como ambiente confiável.

In [5]:
schema_sql = (PROJECT_ROOT / "supabase-schema.sql").read_text(encoding="utf-8").lower()
config_js = (PROJECT_ROOT / "supabase-config.js").read_text(encoding="utf-8").lower()
checks = {
    "RLS ativado na tabela de progresso": "enable row level security" in schema_sql,
    "Bucket de materiais configurado como privado": "'study-materials', 'study-materials', false" in schema_sql,
    "Política de leitura do Storage limitada ao bucket": "bucket_id = 'study-materials'" in schema_sql,
    "Configuração não contém chave service_role": "service_role" not in config_js,
    "Configuração não contém chave sb_secret_": "sb_secret_" not in config_js,
}
for label, passed in checks.items():
    print(f"{'OK' if passed else 'FALHOU'} — {label}")
assert all(checks.values()), "Revise configuração e políticas antes de publicar."

OK — RLS ativado na tabela de progresso
OK — Bucket de materiais configurado como privado
OK — Política de leitura do Storage limitada ao bucket
OK — Configuração não contém chave service_role
OK — Configuração não contém chave sb_secret_


## 5. Desenvolvimento, testes e operação

### Ambiente local

A aplicação não tem etapa de compilação de frontend. Use o Python instalado e um servidor HTTP estático; abrir `index.html` por `file://` pode impedir requisições ao JSON. No PowerShell, na raiz do repositório:

```powershell
python extractor.py
python -m http.server 8000
```

Acesse `http://localhost:8000`. Materiais e arquivos gerados devem permanecer locais ou ser enviados apenas ao bucket privado apropriado.

### Testes automatizados

O extrator usa apenas a biblioteca padrão; a suíte cobre leitura, limpeza de artefatos, alternativas/gabaritos, separação de teoria/questões, deduplicação e geração de índice/flashcards.

```powershell
python -m unittest discover -s tests
```

A próxima célula roda a suíte dentro do notebook e apresenta apenas o resumo do resultado.

### Atualização dos materiais e publicação

1. Atualize os Markdown de origem localmente.
2. Rode `python extractor.py` e revise contagens, questões não identificadas e amostras diretamente no ambiente privado apropriado.
3. Envie `manifest.json`, `question-index.json` e `subjects/*.json` para o bucket privado; não faça commit desses arquivos.
4. Teste o fluxo local de login, carregamento, navegação e sincronização.
5. Publique somente os arquivos estáticos e arquivos públicos necessários. Confira `git status` e `.gitignore` antes de `git add`/push.

### Limitações operacionais

A extração é heurística: fontes com marcadores fora do padrão podem deixar questões na teoria, não reconhecer respostas ou associar assunto/banca incorretamente. Flashcards são sugestões. Confirme dados e gabaritos na fonte original; o material privado não deve ser redistribuído.

In [7]:
import importlib.util
import io
import unittest

test_file = PROJECT_ROOT / "tests" / "test_extractor.py"
spec = importlib.util.spec_from_file_location("project_test_extractor", test_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Não foi possível carregar a suíte: {test_file}")
test_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_module)
suite = unittest.defaultTestLoader.loadTestsFromModule(test_module)
stream = io.StringIO()
result = unittest.TextTestRunner(stream=stream, verbosity=0).run(suite)
print(stream.getvalue().strip())
print(f"Testes executados: {result.testsRun}; erros: {len(result.errors)}; falhas: {len(result.failures)}")
assert result.wasSuccessful(), "A suíte do projeto apresentou falhas; consulte o resultado acima."

----------------------------------------------------------------------
Ran 20 tests in 0.058s

OK
Testes executados: 20; erros: 0; falhas: 0


## 6. Diagnóstico rápido

| Sintoma | O que conferir |
|---|---|
| Página não carrega JSON local | Sirva a pasta por HTTP (`python -m http.server 8000`); não use `file://`. |
| Login não aparece ou não autentica | Verifique URL/chave publishable, usuário permitido, URL de retorno e configuração de autenticação no Supabase. |
| Login funciona, mas materiais não carregam | Confirme que o bucket `study-materials` está privado, que os três grupos de JSON foram enviados nos caminhos corretos e que a política de leitura permite o usuário. |
| Histórico não sincroniza | Verifique conexão, sessão e RLS para `user_progress`; enquanto isso o progresso continua no navegador e a interface informa a falha. |
| Questão ou gabarito não é reconhecido | Revise o marcador numerado, alternativas em linhas reconhecíveis e forma de apresentação do gabarito; valide sempre na fonte original. |
| Mudança de interface não aparece | Confirme o carregamento de `app.js`/`styles.css` e atualize seus parâmetros de versão no HTML para invalidar cache. |
| Teste não encontrado | Execute a descoberta a partir da raiz do projeto: `python -m unittest discover -s tests`. |

### Princípios de manutenção

- Mantenha a extração separada da interface e preserve os JSONs como contrato entre ambas.
- Dê mensagens explícitas para falhas de rede/autenticação; não converta falhas em conteúdo aparentemente vazio.
- Ao alterar o formato JSON, atualize extrator, consumidor JavaScript e testes relacionados na mesma mudança.
- Ao mudar a identidade autorizada, revise cliente e SQL/RLS em conjunto.
- Evite adicionar bibliotecas ao frontend sem necessidade: o site usa HTML/CSS/JavaScript nativos; o extrator usa Python padrão.

## Resumo

O projeto combina geração local de materiais, aplicação estática no navegador e serviços gerenciados pelo Supabase. Os materiais didáticos permanecem privados, enquanto o repositório contém apenas código/documentação. O extrator define o formato que o frontend consome; os testes protegem as etapas de limpeza e extração, e as políticas RLS/Storage impõem o controle de acesso no backend.

In [8]:
ignore_file = PROJECT_ROOT / ".gitignore"
if not ignore_file.is_file():
    raise FileNotFoundError(".gitignore não encontrado na raiz do projeto.")
ignore_rules = ignore_file.read_text(encoding="utf-8")
assert "/data/" in ignore_rules, "A pasta de JSONs gerados precisa ficar fora do Git."
assert "__pycache__/" in ignore_rules, "Cache do Python precisa ficar fora do Git."
notebook_path = PROJECT_ROOT / "documentacao_projeto.ipynb"
assert notebook_path.is_file(), "O notebook ainda não foi salvo no repositório."
print("Privacidade do repositório: data/ e __pycache__/ estão listados no .gitignore.")
print(f"Notebook salvo: {notebook_path.name} ({notebook_path.stat().st_size:,} bytes).")
print("A inspeção não abriu nem imprimiu materiais didáticos ou valores de configuração.")

Privacidade do repositório: data/ e __pycache__/ estão listados no .gitignore.
Notebook salvo: documentacao_projeto.ipynb (0 bytes).
A inspeção não abriu nem imprimiu materiais didáticos ou valores de configuração.
